<p><font size="6" color='grey'> <b>
Machine Learning
</b></font> </br></p>


<p><font size="5" color='grey'> <b>
Neuronale Netze - Keras Regression - Diamonds
</b></font> </br></p>

---


# 0  | Install & Import
***

In [ ]:
# Install

In [ ]:
# Daten
from pandas import read_csv, DataFrame, concat
import numpy as np

# Preprocessing
from sklearn.preprocessing import OrdinalEncoder, StandardScaler

# Modellvorbereitung
from sklearn.model_selection import train_test_split

# Evaluation
from sklearn.metrics import r2_score, mean_absolute_error

# Deep Learning (Keras / TensorFlow)
from keras.layers import Input, Dense
from keras.models import Sequential
from keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from keras.utils import set_random_seed, plot_model

from tensorflow import keras
from tensorflow.config.experimental import enable_op_determinism
import tensorflow as tf

# Visualisierung
import plotly.express as px
import plotly.subplots as sp

In [ ]:
# Warnung ausstellen
import warnings
warnings.filterwarnings("ignore")

# 1 | Understand
---


<p><font color='black' size="5">📋 Checkliste</font></p>

✅ Aufgabe verstehen</br>
✅ Daten sammeln</br>
✅ Statistische Analyse (Min, Max, Mean, Korrelation, ...)</br>
✅ Datenvisualisierung (Streudiagramm, Box-Plot, ...)</br>
✅ Prepare Schritte festlegen</br>

<p><font color='black' size="5">
Anwendungsfall
</font></p>

---

Dieser klassische Datensatz enthält die Preise und andere Attribute von fast 54.000 Diamanten.



[DataSet](https://www.openml.org/search?type=data&status=active&id=42225)

[Info](https://www.kaggle.com/datasets/shivam2503/diamonds)


In [ ]:
df = read_csv(
    "https://raw.githubusercontent.com/ralf-42/ML_Intro/main/02_daten/05_tabellen/diamonds.csv",
    usecols=[
        "carat",
        "cut",
        "color",
        "clarity",
        "depth",
        "table",
        "price",
    ],
)

In [ ]:
data = df.copy()
target = data.pop("price")

<p><font color='black' size="5">
EDA (Exploratory Data Analysis) mit Pandas
</font></p>

In [ ]:
data.info()

In [ ]:
data.describe().T

In [ ]:
data.groupby("cut").count()

In [ ]:
data.groupby("color").count()

<p><font color='black' size="5">
EDA (Exploratory Data Analysis) mit Plotly
</font></p>

In [ ]:
title_ = "Depth"
b1 = px.box(data["depth"], title=title_, width=600, height=600)

title_ = "Carat"
b2 = px.box(data["carat"], title=title_, width=600, height=600)

title_ = "Table"
b3 = px.box(data["table"], title=title_, width=600, height=600)

fig = sp.make_subplots(rows=1, cols=3, subplot_titles=("Depth", "Carat", "Table"))

for trace in b1.data:
    fig.add_trace(trace, row=1, col=1)

for trace in b2.data:
    fig.add_trace(trace, row=1, col=2)

for trace in b3.data:
    fig.add_trace(trace, row=1, col=3)

# Layout anpassen
fig.update_layout(width=1000, height=500, title_text="Box-Plots")

# Plot anzeigen
fig.show()

# 2 |  Prepare

---

<p><font color='black' size="5">📋 Checkliste</font></p>

✅ Datentyp ermitteln/ändern</br>
✅ Train-Test-Split durchführen</br>
✅ Nicht benötigte Features löschen</br>
✅ Missing Values behandeln</br>
✅ Ausreißer behandeln</br>
✅ Kategorischer Features Kodieren</br>
✅ Numerischer Features skalieren</br>
✅ Feature-Engineering (neue Features schaffen)</br>
✅ Dimensionalität reduzieren</br>
✅ Resampling (Over-/Undersampling)</br>
✅ Pipeline erstellen/konfigurieren</br>

<p><font color='black' size="5">
Datentyp ermitteln
</font></p>

In [ ]:
all_col = data.columns
num_col = data.select_dtypes(include="number").columns
cat_col = data.select_dtypes(exclude="number").columns

<p><font color='black' size="5">
Train-Test-Split
</font></p>


In [ ]:
data_train, data_test, target_train, target_test = train_test_split(
    data, target, test_size=0.2, random_state=42)

<p><font color='black' size="5">
Kodierung
</font></p>

In [ ]:
cat_seq = [
    ["Fair", "Good", "Very Good", "Premium", "Ideal"],
    ["J", "I", "H", "G", "F", "E", "D"],
    ["I1", "SI2", "SI1", "VS2", "VS1", "VVS2", "VVS1", "IF"],]

encoder = OrdinalEncoder(categories=cat_seq)
encoder.fit(data_train[cat_col])
data_train[cat_col] = encoder.transform(data_train[cat_col])
data_test[cat_col] = encoder.transform(data_test[cat_col])

<font color='black' size="5">
Skalierung
</font>

In [ ]:
scaler = StandardScaler()
scaler.fit(data_train[all_col])
data_train[all_col] = scaler.transform(data_train[all_col])
data_test[all_col] = scaler.transform(data_test[all_col])

# 3 | Modeling
---

<p><font color='black' size="5">📋 Checkliste</font></p>

✅ Modellauswahl treffen</br>
✅ Pipeline erweitern/konfigurieren</br>
✅ Training durchführen</br>
✅ Hyperparameter Tuning</br>
✅ Cross-Validiation</br>
✅ Bootstrapping</br>
✅ Regularization</br>

<p><font color='black' size="5">🧠 Algorithmus: Keras Sequential – Neuronales Netz (Regression)</font></p>

Keras ist eine hochstufige API für den Aufbau neuronaler Netze. Bei Regressions-problemen hat die letzte Schicht keine Aktivierungsfunktion oder verwendet `linear` – damit kann das Modell beliebige Zahlenwerte vorhersagen (z.B. einen Preis).

**Analogie:** Wie ein Fließband – jede Schicht verarbeitet die Daten weiter, bis am Ende kein Label, sondern ein Messwert herauskommt.

**Wichtige Konzepte:**

| Begriff | Bedeutung |
|---------|----------|
| `Dense` | Vollständig verbundene Schicht |
| `optimizer` | Optimierungsalgorithmus (z.B. Adam) |
| `loss` | Verlustfunktion (z.B. `mse` – Mean Squared Error) |
| `epochs` | Anzahl der Trainingsdurchläufe |

**In der Praxis relevant wenn:**
- Kontinuierliche Zahlenwerte vorhergesagt werden und eine präzise Kontrolle über Netzwerkarchitektur und Trainingsprozess (Callbacks, Loss, Optimizer) nötig ist
- Callbacks wie EarlyStopping oder Modell-Checkpoints eingesetzt werden sollen, um das beste Modell zu sichern
- Komplexe nichtlineare Preisstrukturen modelliert werden sollen, die über lineare Zusammenhänge hinausgehen

**Nicht geeignet wenn:**
- Einfache Regressionsaufgaben mit klar linearen Zusammenhängen vorliegen → dann LinearRegression oder Ridge
- Der Datensatz sehr klein ist → neuronale Netze neigen dann zu Overfitting

**Typischer Fehler:**
Für Regression `sigmoid` als Ausgabe-Aktivierung verwenden — `sigmoid` begrenzt die Ausgabe auf [0, 1]. Die letzte Dense-Schicht bei Regression hat keine Aktivierungsfunktion (linear) oder explizit `activation="linear"`.

In [ ]:
#@markdown   <p><font size="4" color='green'>  Keras Sequential – Regression</font> </br></p>

import base64
from IPython.display import Image, display

diagram = """
flowchart TD
    InputLayer["Input Layer<br/>Input: 6 Features"]
    Hidden1["Dense Layer 1<br/>Units: 88 | Activation: ReLU"]
    Hidden2["Dense Layer 2<br/>Units: 44 | Activation: ReLU"]
    Hidden3["Dense Layer 3<br/>Units: 11 | Activation: ReLU"]
    OutputLayer["Output Layer<br/>Units: 1 | Activation: Linear"]
    InputLayer --> |6 Features| Hidden1
    Hidden1 --> |88 Signale| Hidden2
    Hidden2 --> |44 Signale| Hidden3
    Hidden3 --> |11 Signale| OutputLayer
    style InputLayer fill:#f9f,stroke:#333,stroke-width:2px
    style Hidden1 fill:#bbf,stroke:#333,stroke-width:2px
    style Hidden2 fill:#bbf,stroke:#333,stroke-width:2px
    style Hidden3 fill:#bbf,stroke:#333,stroke-width:2px
    style OutputLayer fill:#bfb,stroke:#333,stroke-width:2px
"""

encoded = base64.urlsafe_b64encode(diagram.strip().encode()).decode()
display(Image(url=f"https://mermaid.ink/img/{encoded}", width=250))

<p><font color='black' size="5">
Zufallszahl initialisieren
</font></p>

In [ ]:
set_random_seed(42)
enable_op_determinism()

<p><font color='black' size="5">
Aufbau des Neuronale Netzes
</font></p>


In [ ]:
model = Sequential()
model.add(Input(shape=(6,)))
model.add(Dense(units=88, activation="relu"))
model.add(Dense(units=44, activation="relu"))
model.add(Dense(units=11, activation="relu"))
model.add(Dense(1))

In [ ]:
# Visualisierung neuronales Netz
model.summary()

In [ ]:
# Visualisierung neuronales Netz
plot_model(
    model,
    to_file="nn_structure.png",
    show_shapes=True,
    show_dtype=True,
    show_layer_names=True,
    dpi=100,
    expand_nested=True,
    show_layer_activations=True
)

In [ ]:
# Anzahl Parameter je Layer
for layer in model.layers:
    print(f"{layer.name}: {layer.count_params()} Parameter")

<p><font color='black' size="5">
Compile
</font></p>



In [ ]:
model.compile(optimizer="adam", loss="mae")

<p><font color='black' size="5">
Callback - deaktiviert
</font></p>

In [ ]:
# early = EarlyStopping(monitor="loss", patience=5)

# reduce_lr = ReduceLROnPlateau(
#     monitor="loss",
#     factor=0.1,
#     patience=2,
#     verbose=1,
#     mode="auto",
#     min_delta=0.0001,
#     cooldown=0,
#     min_lr=0,
# )

# check = ModelCheckpoint(filepath="model.keras", monitor="loss", save_best_only=True)

<p><font color='black' size="5">
Training
</font></p>

In [ ]:
model.fit(
    data_train,
    target_train,
    epochs=20,
    validation_split=0.2,
    # callbacks=[early, reduce_lr, check],
)

In [ ]:
print(model.history.params)
print(model.history.history.keys())
save_history = model.history.history.keys()

<p><font color='black' size="5">
Loss-Entwickung
</font></p>

In [ ]:
title_ = "Loss-Entwicklung"
px.line(
    y=model.history.history["loss"],
    title=title_,
    labels={"x": "Epochen", "y": "Loss-Wert"},
    width=800,
    height=400,
)

# 4 | Evaluate
---

<p><font color='black' size="5">📋 Checkliste</font></p>

✅ Prognose (Train, Test) erstellen</br>
✅ Modellgüte prüfen</br>
✅ Residuenanalyse erstellen</br>
✅ Feature Importance/Selektion prüfen</br>
✅ Robustheitstest erstellen</br>
✅ Modellinterpretation erstellen</br>
✅ Sensitivitätsanalyse erstellen</br>
✅ Kommunikation (Key Takeaways)</br>


<p><font color='black' size="5">
Prediction
</font></p>


In [ ]:
target_train_pred = model.predict(data_train)
target_test_pred = model.predict(data_test)

<p><font color='black' size="5">
Bestimmtheitsmass
</font></p>

In [ ]:
r2 = r2_score(target_train, target_train_pred)
print(f"Modell: {model} -- Train --- Bestimmtheitsmass: {r2:5.2f}")

In [ ]:
r2 = r2_score(target_test, target_test_pred)
print(f"Modell: {model} -- Test --- Bestimmtheitsmass: {r2:5.2f}")

<p><font color='black' size="5">
Mean Absolut Error
</font></p>

In [ ]:
mae = mean_absolute_error(target_test, target_test_pred)
print(f"Modell: {model} -- Test -- Mean Absolute Error: {mae:5.2f}")

<p><font color='black' size="5">
Aufbau Analysewürfel
</font></p>

In [ ]:
# Übernahme der Testdaten
cube = data_test.copy()
cube.reset_index(inplace=True)

# Übernahme Target real & predict
cube["real"]    = target_test.values
cube["predict"] = target_test_pred.ravel()

<p><font color='black' size="5">
Visualisierung real vs predict
</font></p>

In [ ]:
# Boxplot
title_ = "Boxplot real vs predict"
px.box(cube[["real", "predict"]], title=title_, width=600, height=600)

In [ ]:
# Histogramm
title_ = "Histogramm Prices real vs predict"
fig = px.histogram(cube, x=["real", "predict"], nbins=10, text_auto=".2s", title=title_)
fig.update_layout(barmode="group", bargap=0.2, width=800, height=600)
fig.show()

<p><font color='black' size="5">
Fehlerhafte Vorhersagen
</font></p>

In [ ]:
cube["abs_Abw%"] = abs((cube["real"] - cube["predict"]) / cube["real"] * 100)
cube.head(10).style.format("{:,.1f}")

In [ ]:
cube.describe().T

In [ ]:
# Histogramm
title_ = "Histogramm absolute Abweichung"
fig = px.histogram(cube, x=["abs_Abw%"], nbins=20, text_auto=".2s", title=title_)
fig.update_layout(barmode="group", bargap=0.2, width=800, height=600)
fig.show()

<p><font color='black' size="5">
Residuals Plot
</font></p>

In [ ]:
#@title
#@markdown   <p><font size="4" color='green'>  Plot-Funtion, für Keras</font> </br></p>
def erstelle_residuals_plot(target_train, target_train_pred, target_test, target_test_pred, title):
    """Erstellt Residuals Plot mit Train/Test Split"""
    # Arrays zu 1D konvertieren
    target_train = np.array(target_train).ravel()
    target_train_pred = np.array(target_train_pred).ravel()
    target_test = np.array(target_test).ravel()
    target_test_pred = np.array(target_test_pred).ravel()

    df_train = DataFrame({
        'Vorhersage': target_train_pred,
        'Residuen': target_train_pred - target_train,
        'Set': 'Train'
    })

    df_test = DataFrame({
        'Vorhersage': target_test_pred,
        'Residuen': target_test_pred - target_test,
        'Set': 'Test'
    })

    df = concat([df_train, df_test])

    fig = px.scatter(df, x='Vorhersage', y='Residuen',
                    color='Set',
                    color_discrete_map={'Train': 'blue', 'Test': 'green'},
                    opacity=0.5,  # Transparenz hinzugefügt
                    title=title,
                    labels={'Vorhersage': 'Vorhergesagte Werte',
                           'Residuen': 'Residuen'},
                    marginal_y='histogram',
                    width=1000,   # Breite in Pixeln
                    height=600)

    fig.add_hline(y=0, line_color="black")
    fig.update_layout(height=600, margin=dict(t=100))
    fig.update_layout(
        xaxis=dict(domain=[0, 0.85]),  # Scatter nimmt 85% der Breite ein
        xaxis2=dict(domain=[0.86, 1]),
        title='Häufigkeit'
        )

    return fig

In [ ]:
erstelle_residuals_plot(target_train, target_train_pred, target_test, target_test_pred, "Residuals Plot Model keras")

# 5 | Deploy
---

<p><font color='black' size="5">📋 Checkliste</font></p>

✅ Modellexport und -speicherung</br>
✅ Abhängigkeiten und Umgebung</br>
✅ Sicherheit und Datenschutz</br>
✅ In die Produktion integrieren</br>
✅ Tests und Validierung</br>
✅ Dokumentation & Wartung</br>